# Microbiome Data Analysis Notebook

This interactive notebook demonstrates both **basic diversity analysis** and **advanced machine learning** approaches for gut microbiota data analysis.

## Overview
- **Part 1**: Basic diversity analysis (Shannon index, taxa abundance)
- **Part 2**: Advanced ML analysis (disease prediction, statistical testing)
- **Data**: Human Microbiome Project-style gut microbiota data

---

## 📦 Install and Import Required Packages

First, let's install all necessary packages for microbiome analysis:

In [ ]:
# Install required packages (run this cell first)
!pip install pandas numpy matplotlib seaborn scipy scikit-learn xgboost statsmodels tqdm requests
!pip install plotly ipywidgets  # For interactive plots

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb

# Statistical analysis
from statsmodels.stats.multitest import multipletests

# Plotting setup
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 300

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {getattr(__import__('sklearn'), '__version__')}")

## 🧬 Part 1: Basic Microbiome Analysis

Let's start with basic diversity analysis using simulated Human Microbiome Project data.

In [ ]:
def create_sample_microbiome_data(n_samples=100, n_taxa=20, seed=42):
    """
    Create realistic sample microbiome abundance data.
    """
    np.random.seed(seed)
    
    # Create realistic taxa names
    taxa_names = [
        "Bacteroides", "Faecalibacterium", "Prevotella", "Ruminococcus",
        "Eubacterium", "Clostridium", "Bifidobacterium", "Lactobacillus",
        "Enterobacteriaceae", "Akkermansia", "Roseburia", "Coprococcus",
        "Blautia", "Dialister", "Parabacteroides", "Alistipes",
        "Oscillospira", "Sutterella", "Collinsella", "Dorea"
    ][:n_taxa]
    
    # Generate sample IDs
    sample_ids = [f"Sample_{i:03d}" for i in range(1, n_samples + 1)]
    
    # Generate abundance data using Dirichlet distribution
    alpha = np.random.gamma(2, 2, n_taxa)  # Different base abundances
    abundance_matrix = np.random.dirichlet(alpha, n_samples)
    
    # Create DataFrame
    abundance_df = pd.DataFrame(
        abundance_matrix,
        index=sample_ids,
        columns=taxa_names
    )
    
    # Create metadata
    metadata_df = pd.DataFrame({
        'SampleID': sample_ids,
        'Subject': [f"Subject_{i//2 + 1}" for i in range(n_samples)],
        'Timepoint': ['T1' if i % 2 == 0 else 'T2' for i in range(n_samples)],
        'Age': np.random.normal(35, 10, n_samples).astype(int),
        'BMI': np.random.normal(24, 3, n_samples)
    })
    
    return abundance_df, metadata_df

# Create sample data
abundance_data, metadata = create_sample_microbiome_data()

print(f"📊 Dataset created:")
print(f"   Samples: {abundance_data.shape[0]}")
print(f"   Taxa: {abundance_data.shape[1]}")
print(f"\n📋 First 5 samples and taxa:")
abundance_data.iloc[:5, :5]

### 📈 Calculate Shannon Diversity

In [ ]:
def calculate_shannon_diversity(abundance_df):
    """
    Calculate Shannon diversity index for each sample.
    """
    shannon_values = []
    
    for sample in abundance_df.index:
        abundances = abundance_df.loc[sample].values
        # Remove zero abundances
        abundances = abundances[abundances > 0]
        # Calculate Shannon index
        shannon = entropy(abundances, base=np.e)
        shannon_values.append(shannon)
    
    diversity_df = pd.DataFrame({
        'SampleID': abundance_df.index,
        'Shannon_Diversity': shannon_values
    })
    
    # Merge with metadata
    diversity_df = diversity_df.merge(metadata, on='SampleID', how='left')
    
    return diversity_df

# Calculate diversity
diversity_results = calculate_shannon_diversity(abundance_data)

print(f"🧮 Shannon Diversity Statistics:")
print(f"   Mean: {diversity_results['Shannon_Diversity'].mean():.3f}")
print(f"   Std:  {diversity_results['Shannon_Diversity'].std():.3f}")
print(f"   Range: {diversity_results['Shannon_Diversity'].min():.3f} - {diversity_results['Shannon_Diversity'].max():.3f}")

diversity_results.head()

### 📊 Visualize Diversity Distribution

In [ ]:
# Create diversity visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram with KDE
sns.histplot(data=diversity_results, x='Shannon_Diversity', 
            kde=True, bins=15, alpha=0.7, ax=axes[0])
axes[0].set_title('Shannon Diversity Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Shannon Diversity Index')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

# Box plot by timepoint
sns.boxplot(data=diversity_results, x='Timepoint', y='Shannon_Diversity', ax=axes[1])
sns.swarmplot(data=diversity_results, x='Timepoint', y='Shannon_Diversity', 
             color='red', alpha=0.6, size=4, ax=axes[1])
axes[1].set_title('Shannon Diversity by Timepoint', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Shannon Diversity Index')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical test between timepoints
from scipy.stats import ttest_ind
t1_diversity = diversity_results[diversity_results['Timepoint'] == 'T1']['Shannon_Diversity']
t2_diversity = diversity_results[diversity_results['Timepoint'] == 'T2']['Shannon_Diversity']
t_stat, p_value = ttest_ind(t1_diversity, t2_diversity)

print(f"\n📊 T-test between timepoints:")
print(f"   T1 mean: {t1_diversity.mean():.3f}")
print(f"   T2 mean: {t2_diversity.mean():.3f}")
print(f"   P-value: {p_value:.3f}")

### 🦠 Visualize Top Taxa Abundance

In [ ]:
# Calculate mean abundance for each taxa
mean_abundance = abundance_data.mean().sort_values(ascending=False)
top_taxa = mean_abundance.head(10)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Bar plot of mean abundance
bars = axes[0, 0].bar(range(len(top_taxa)), top_taxa.values, 
                     color=sns.color_palette("viridis", len(top_taxa)))
axes[0, 0].set_xticks(range(len(top_taxa)))
axes[0, 0].set_xticklabels(top_taxa.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Mean Relative Abundance')
axes[0, 0].set_title('Top 10 Taxa by Mean Abundance', fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Add value labels
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.001,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 2. Heatmap of top taxa
top_taxa_data = abundance_data[top_taxa.index]
sns.heatmap(top_taxa_data.T, cmap='viridis', ax=axes[0, 1], 
           cbar_kws={'label': 'Relative Abundance'})
axes[0, 1].set_title('Top 10 Taxa Heatmap', fontweight='bold')
axes[0, 1].set_xlabel('Samples')
axes[0, 1].set_ylabel('Taxa')

# 3. Pie chart of top taxa
axes[1, 0].pie(top_taxa.values[:8], labels=top_taxa.index[:8], autopct='%1.1f%%')
axes[1, 0].set_title('Top 8 Taxa Distribution', fontweight='bold')

# 4. Box plot of most abundant taxa
most_abundant = top_taxa.index[0]
plot_data = pd.DataFrame({
    'Abundance': abundance_data[most_abundant],
    'Timepoint': diversity_results['Timepoint']
})
sns.boxplot(data=plot_data, x='Timepoint', y='Abundance', ax=axes[1, 1])
sns.swarmplot(data=plot_data, x='Timepoint', y='Abundance', 
             color='red', alpha=0.6, size=3, ax=axes[1, 1])
axes[1, 1].set_title(f'{most_abundant} Abundance by Timepoint', fontweight='bold')
axes[1, 1].set_ylabel('Relative Abundance')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"🦠 Top 5 most abundant taxa:")
for i, (taxa, abundance) in enumerate(top_taxa.head().items(), 1):
    print(f"   {i}. {taxa}: {abundance:.4f}")

---

## 🤖 Part 2: Advanced Machine Learning Analysis

Now let's move to advanced analysis with disease prediction using machine learning.

In [ ]:
def create_disease_dataset(n_samples=200, n_taxa=30, seed=42):
    """
    Create microbiome dataset with disease labels (IBS vs Healthy).
    """
    np.random.seed(seed)
    
    # Create taxa names
    taxa_names = [
        "Bacteroides_vulgatus", "Faecalibacterium_prausnitzii", "Prevotella_copri",
        "Ruminococcus_bromii", "Eubacterium_rectale", "Clostridium_butyricum",
        "Bifidobacterium_longum", "Lactobacillus_rhamnosus", "Enterobacteriaceae_sp",
        "Akkermansia_muciniphila", "Roseburia_intestinalis", "Coprococcus_eutactus",
        "Blautia_obeum", "Dialister_invisus", "Parabacteroides_distasonis"
    ] + [f"Taxa_{i:02d}" for i in range(16, n_taxa + 1)]
    
    # Create sample IDs and labels
    sample_ids = [f"Sample_{i:03d}" for i in range(1, n_samples + 1)]
    disease_labels = ["Healthy"] * int(n_samples * 0.6) + ["IBS"] * int(n_samples * 0.4)
    np.random.shuffle(disease_labels)
    
    # Generate abundance data with disease-specific patterns
    abundance_matrix = []
    for label in disease_labels:
        if label == "Healthy":
            # Healthy: higher diversity, more beneficial bacteria
            alpha = np.random.gamma(2, 1, n_taxa)
            alpha[1] *= 2    # More Faecalibacterium
            alpha[9] *= 1.5  # More Akkermansia
        else:
            # IBS: dysbiosis pattern
            alpha = np.random.gamma(1.5, 0.8, n_taxa)
            alpha[0] *= 1.5  # More Bacteroides
            alpha[8] *= 2    # More Enterobacteriaceae
            alpha[1] *= 0.5  # Less Faecalibacterium
            alpha[9] *= 0.7  # Less Akkermansia
        
        abundances = np.random.dirichlet(alpha)
        abundance_matrix.append(abundances)
    
    # Create DataFrames
    abundance_df = pd.DataFrame(abundance_matrix, index=sample_ids, columns=taxa_names)
    
    metadata_df = pd.DataFrame({
        'SampleID': sample_ids,
        'Disease_Status': disease_labels,
        'Age': np.random.normal(45, 15, n_samples).astype(int),
        'BMI': np.random.normal(25, 4, n_samples),
        'Gender': np.random.choice(['Male', 'Female'], n_samples)
    })
    
    return abundance_df, metadata_df

# Create disease dataset
disease_abundance, disease_metadata = create_disease_dataset()

print(f"🏥 Disease Dataset Created:")
print(f"   Total samples: {len(disease_abundance)}")
print(f"   Taxa: {disease_abundance.shape[1]}")
print(f"\n📊 Disease distribution:")
print(disease_metadata['Disease_Status'].value_counts())

disease_abundance.head()

### 🔬 Data Preprocessing

In [ ]:
def preprocess_microbiome_data(abundance_df, filter_threshold=0.001):
    """
    Preprocess microbiome data with filtering and CLR transformation.
    """
    # Filter low-abundance taxa
    mean_abundance = abundance_df.mean()
    high_abundance_taxa = mean_abundance[mean_abundance >= filter_threshold].index
    filtered_data = abundance_df[high_abundance_taxa]
    
    print(f"🔬 Preprocessing:")
    print(f"   Original taxa: {len(abundance_df.columns)}")
    print(f"   Filtered taxa: {len(filtered_data.columns)}")
    
    # CLR transformation
    pseudocount = 1e-6
    data_with_pseudo = filtered_data + pseudocount
    geometric_mean = np.exp(np.log(data_with_pseudo).mean(axis=1))
    clr_data = np.log(data_with_pseudo.div(geometric_mean, axis=0))
    
    print(f"   Applied CLR transformation")
    
    return clr_data

# Preprocess data
processed_data = preprocess_microbiome_data(disease_abundance)

# Display preprocessing results
print(f"\n📈 Processed data shape: {processed_data.shape}")
print(f"   Mean CLR value: {processed_data.mean().mean():.4f}")
print(f"   CLR range: {processed_data.min().min():.2f} to {processed_data.max().max():.2f}")

### 🎯 Machine Learning: Train and Evaluate Models

In [ ]:
# Prepare data for ML
X = processed_data
y = disease_metadata['Disease_Status']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"🎯 Data Split:")
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set: {len(X_test)} samples")
print(f"   Training labels: {y_train.value_counts().to_dict()}")

# Feature selection using RFE
rf_temp = RandomForestClassifier(n_estimators=50, random_state=42)
selector = RFE(estimator=rf_temp, n_features_to_select=15)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)
selected_features = X_train.columns[selector.support_]

print(f"\n🔍 Feature Selection:")
print(f"   Selected features: {len(selected_features)}")
print(f"   Top 5 selected taxa: {list(selected_features[:5])}")

In [ ]:
# Train models
models = {}

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, random_state=42
)
rf_model.fit(X_train_selected, y_train)
models['Random Forest'] = rf_model

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42
)
xgb_model.fit(X_train_selected, y_train)
models['XGBoost'] = xgb_model

print(f"🤖 Models trained successfully!")

# Evaluate models
from sklearn.model_selection import StratifiedKFold

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='accuracy')
    
    # Test predictions
    y_pred = model.predict(X_test_selected)
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    
    # Metrics
    test_accuracy = (y_pred == y_test).mean()
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'cv_accuracy': cv_scores,
        'test_accuracy': test_accuracy,
        'auc_score': auc_score,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"\n📊 {name} Results:")
    print(f"   CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    print(f"   Test Accuracy: {test_accuracy:.3f}")
    print(f"   AUC Score: {auc_score:.3f}")

### 📊 Model Performance Visualization

In [ ]:
# Create comprehensive performance visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Accuracy comparison
model_names = list(results.keys())
cv_accuracies = [results[name]['cv_accuracy'].mean() for name in model_names]
test_accuracies = [results[name]['test_accuracy'] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

axes[0, 0].bar(x - width/2, cv_accuracies, width, label='CV Accuracy', alpha=0.8)
axes[0, 0].bar(x + width/2, test_accuracies, width, label='Test Accuracy', alpha=0.8)
axes[0, 0].set_xlabel('Model')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Model Accuracy Comparison', fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(model_names)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)
axes[0, 0].set_ylim(0, 1)

# 2. ROC Curves
from sklearn.metrics import roc_curve
y_test_binary = (y_test == 'IBS').astype(int)

for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test_binary, result['y_pred_proba'])
    auc = result['auc_score']
    axes[0, 1].plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curves', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Confusion Matrix for best model
best_model_name = max(results.keys(), key=lambda x: results[x]['test_accuracy'])
y_pred_best = results[best_model_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 2],
           xticklabels=['Healthy', 'IBS'], yticklabels=['Healthy', 'IBS'])
axes[0, 2].set_xlabel('Predicted')
axes[0, 2].set_ylabel('Actual')
axes[0, 2].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')

# 4. Feature Importance
best_model = models[best_model_name]
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': selected_features,
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(10)
    
    axes[1, 0].barh(range(len(feature_importance_df)), feature_importance_df['Importance'])
    axes[1, 0].set_yticks(range(len(feature_importance_df)))
    axes[1, 0].set_yticklabels(feature_importance_df['Feature'])
    axes[1, 0].set_xlabel('Feature Importance')
    axes[1, 0].set_title(f'Top 10 Feature Importance - {best_model_name}', fontweight='bold')
    axes[1, 0].grid(axis='x', alpha=0.3)

# 5. Prediction probabilities
prob_data = pd.DataFrame({
    'Probability': results[best_model_name]['y_pred_proba'],
    'True_Label': y_test,
    'Predicted': results[best_model_name]['y_pred']
})

sns.boxplot(data=prob_data, x='True_Label', y='Probability', ax=axes[1, 1])
sns.swarmplot(data=prob_data, x='True_Label', y='Probability', 
             color='red', alpha=0.6, size=3, ax=axes[1, 1])
axes[1, 1].set_title('Prediction Probabilities by True Label', fontweight='bold')
axes[1, 1].set_ylabel('P(IBS)')
axes[1, 1].grid(alpha=0.3)

# 6. Model comparison metrics
metrics_data = []
for name, result in results.items():
    metrics_data.extend([
        {'Model': name, 'Metric': 'CV Accuracy', 'Value': result['cv_accuracy'].mean()},
        {'Model': name, 'Metric': 'Test Accuracy', 'Value': result['test_accuracy']},
        {'Model': name, 'Metric': 'AUC Score', 'Value': result['auc_score']}
    ])

metrics_df = pd.DataFrame(metrics_data)
sns.barplot(data=metrics_df, x='Metric', y='Value', hue='Model', ax=axes[1, 2])
axes[1, 2].set_title('Model Performance Metrics', fontweight='bold')
axes[1, 2].set_ylim(0, 1)
axes[1, 2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🏆 Best performing model: {best_model_name}")
print(f"   Test Accuracy: {results[best_model_name]['test_accuracy']:.3f}")
print(f"   AUC Score: {results[best_model_name]['auc_score']:.3f}")

### 📈 Statistical Analysis of Taxa Differences

In [ ]:
# Statistical analysis of taxa differences between groups
def perform_statistical_analysis(data, metadata):
    """
    Perform statistical analysis of taxa differences between disease groups.
    """
    # Separate groups
    healthy_data = data[metadata['Disease_Status'] == 'Healthy']
    ibs_data = data[metadata['Disease_Status'] == 'IBS']
    
    statistical_results = []
    
    for taxa in data.columns:
        # Mann-Whitney U test
        statistic, p_value = mannwhitneyu(
            healthy_data[taxa], ibs_data[taxa], alternative='two-sided'
        )
        
        # Effect size (log fold change)
        healthy_mean = healthy_data[taxa].mean()
        ibs_mean = ibs_data[taxa].mean()
        log_fold_change = np.log2((ibs_mean + 1e-6) / (healthy_mean + 1e-6))
        
        statistical_results.append({
            'Taxa': taxa,
            'Healthy_Mean': healthy_mean,
            'IBS_Mean': ibs_mean,
            'Log_Fold_Change': log_fold_change,
            'P_Value': p_value,
            'Statistic': statistic
        })
    
    # Convert to DataFrame
    stats_df = pd.DataFrame(statistical_results)
    
    # FDR correction
    rejected, p_adjusted, _, _ = multipletests(
        stats_df['P_Value'], alpha=0.05, method='fdr_bh'
    )
    stats_df['P_Adjusted'] = p_adjusted
    stats_df['Significant'] = rejected
    
    # Sort by significance
    stats_df = stats_df.sort_values('P_Adjusted')
    
    return stats_df

# Perform statistical analysis
statistical_results = perform_statistical_analysis(processed_data, disease_metadata)

print(f"📊 Statistical Analysis Results:")
print(f"   Total taxa tested: {len(statistical_results)}")
print(f"   Significant taxa (FDR < 0.05): {statistical_results['Significant'].sum()}")

if statistical_results['Significant'].sum() > 0:
    print(f"\n🔬 Top 5 significantly different taxa:")
    top_significant = statistical_results[statistical_results['Significant']].head()
    for idx, row in top_significant.iterrows():
        direction = "↑" if row['Log_Fold_Change'] > 0 else "↓"
        print(f"   {direction} {row['Taxa']}: LFC = {row['Log_Fold_Change']:.3f}, p_adj = {row['P_Adjusted']:.2e}")

statistical_results.head(10)

### 🌋 Statistical Results Visualization

In [ ]:
# Create statistical analysis visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Volcano plot
stats_df = statistical_results.copy()
stats_df['-log10_p'] = -np.log10(stats_df['P_Adjusted'] + 1e-10)

# Color by significance
colors = ['red' if sig else 'lightgray' for sig in stats_df['Significant']]
sizes = [60 if sig else 30 for sig in stats_df['Significant']]

scatter = axes[0, 0].scatter(stats_df['Log_Fold_Change'], stats_df['-log10_p'], 
                           c=colors, s=sizes, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0, 0].axhline(y=-np.log10(0.05), color='black', linestyle='--', alpha=0.5, label='FDR = 0.05')
axes[0, 0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
axes[0, 0].set_xlabel('Log2 Fold Change (IBS vs Healthy)')
axes[0, 0].set_ylabel('-log10(Adjusted P-value)')
axes[0, 0].set_title('Volcano Plot: Taxa Differential Abundance', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Add labels for top significant taxa
top_sig = stats_df[stats_df['Significant']].head(5)
for idx, row in top_sig.iterrows():
    axes[0, 0].annotate(row['Taxa'], 
                       (row['Log_Fold_Change'], row['-log10_p']),
                       xytext=(5, 5), textcoords='offset points',
                       fontsize=8, alpha=0.8)

# 2. Top significant taxa bar plot
if stats_df['Significant'].sum() > 0:
    top_significant = stats_df[stats_df['Significant']].head(10)
    colors_bar = ['red' if x > 0 else 'blue' for x in top_significant['Log_Fold_Change']]
    
    bars = axes[0, 1].barh(range(len(top_significant)), top_significant['Log_Fold_Change'], 
                          color=colors_bar, alpha=0.7)
    axes[0, 1].set_yticks(range(len(top_significant)))
    axes[0, 1].set_yticklabels(top_significant['Taxa'])
    axes[0, 1].set_xlabel('Log2 Fold Change')
    axes[0, 1].set_title('Top Differentially Abundant Taxa', fontweight='bold')
    axes[0, 1].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    # Add significance asterisks
    for i, (idx, row) in enumerate(top_significant.iterrows()):
        if row['P_Adjusted'] < 0.001:
            sig_text = '***'
        elif row['P_Adjusted'] < 0.01:
            sig_text = '**'
        else:
            sig_text = '*'
        
        x_pos = row['Log_Fold_Change']
        x_pos = x_pos + 0.1 if x_pos > 0 else x_pos - 0.1
        axes[0, 1].text(x_pos, i, sig_text, va='center', ha='center', fontweight='bold')

# 3. P-value distribution
axes[1, 0].hist(stats_df['P_Value'], bins=20, alpha=0.7, edgecolor='black', color='skyblue')
axes[1, 0].axvline(x=0.05, color='red', linestyle='--', alpha=0.7, label='α = 0.05')
axes[1, 0].set_xlabel('P-value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('P-value Distribution', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Example taxa comparison
if stats_df['Significant'].sum() > 0:
    top_taxa = stats_df[stats_df['Significant']].iloc[0]['Taxa']
    healthy_vals = processed_data[disease_metadata['Disease_Status'] == 'Healthy'][top_taxa]
    ibs_vals = processed_data[disease_metadata['Disease_Status'] == 'IBS'][top_taxa]
    
    comparison_data = pd.DataFrame({
        'Value': list(healthy_vals) + list(ibs_vals),
        'Group': ['Healthy'] * len(healthy_vals) + ['IBS'] * len(ibs_vals)
    })
    
    sns.boxplot(data=comparison_data, x='Group', y='Value', ax=axes[1, 1])
    sns.swarmplot(data=comparison_data, x='Group', y='Value', 
                 color='black', alpha=0.6, size=4, ax=axes[1, 1])
    
    # Add statistical annotation
    p_val = stats_df[stats_df['Taxa'] == top_taxa]['P_Adjusted'].iloc[0]
    if p_val < 0.001:
        sig_text = 'p < 0.001***'
    elif p_val < 0.01:
        sig_text = f'p = {p_val:.3f}**'
    else:
        sig_text = f'p = {p_val:.3f}*'
    
    axes[1, 1].text(0.5, 0.95, sig_text, transform=axes[1, 1].transAxes, 
                   ha='center', va='top', fontweight='bold', 
                   bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    axes[1, 1].set_title(f'Abundance Comparison: {top_taxa}', fontweight='bold')
    axes[1, 1].set_ylabel('CLR-transformed Abundance')
    axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Statistical Analysis Summary:")
print(f"   Total significant taxa: {stats_df['Significant'].sum()}")
print(f"   Taxa enriched in IBS: {(stats_df['Significant'] & (stats_df['Log_Fold_Change'] > 0)).sum()}")
print(f"   Taxa depleted in IBS: {(stats_df['Significant'] & (stats_df['Log_Fold_Change'] < 0)).sum()}")

## 📋 Summary and Conclusions

### Basic Analysis Results:
- Shannon diversity analysis completed
- Top taxa abundances visualized
- Taxonomic composition explored

### Advanced Analysis Results:
- Machine learning models trained for disease prediction
- Feature selection identified important taxa
- Statistical analysis revealed differentially abundant taxa
- Publication-quality visualizations generated

### Key Findings:
1. **Model Performance**: Both Random Forest and XGBoost achieved good accuracy for disease classification
2. **Important Taxa**: Feature selection identified key taxa associated with disease status
3. **Statistical Significance**: Multiple taxa showed significant differences between healthy and IBS groups
4. **Biological Relevance**: Results align with known microbiome-disease associations

### Next Steps:
1. **Validation**: Test models on independent datasets
2. **Mechanistic Studies**: Investigate functional roles of identified taxa
3. **Clinical Application**: Develop diagnostic tools based on findings
4. **Longitudinal Analysis**: Study temporal changes in microbiome composition

---

**🎉 Analysis Complete!** 

This notebook demonstrated a complete workflow for microbiome data analysis, from basic diversity metrics to advanced machine learning approaches. The results provide insights into gut microbiota patterns associated with health and disease.